# H3-B baseline_whitening — test inference

Frozen test prompts for Devign and ReVeal (context 8192 UTF-8 bytes). This package sends the saved prompts to DeepSeek V4.1; it does not rebuild graph evidence or expose test labels.

Run the setup script once, select the `Python (baseline-whitening-test)` kernel, then execute the configuration cell to see the selected model and endpoint. `PERCENT` controls the prefix cohort; start at 5% and set it to 100% for the complete split.


In [5]:
import os
import sys
from pathlib import Path

BUNDLE_FOLDER = "SHARE_BASELINE_WHITENING_DEVIGN_REVEAL_TEST_CONTEXT8192_20261024"

def find_bundle():
    configured = os.environ.get("BASELINE_WHITENING_BUNDLE_ROOT", "").strip()
    candidates = [Path(configured).expanduser()] if configured else []
    cwd = Path.cwd().resolve()
    for parent in (cwd, *cwd.parents):
        candidates.extend([parent, parent / BUNDLE_FOLDER])
    candidates.extend([Path(r"D:\JUNIOR\LabThayTho") / BUNDLE_FOLDER, Path("/mnt/d/JUNIOR/LabThayTho") / BUNDLE_FOLDER])
    seen = set()
    for candidate in candidates:
        try:
            candidate = candidate.resolve()
        except OSError:
            continue
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "runner.py").is_file() and (candidate / "bundle_manifest.json").is_file():
            return candidate
    raise FileNotFoundError("Cannot find test inference bundle; set BASELINE_WHITENING_BUNDLE_ROOT or open from the package folder")

ROOT = find_bundle()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runner import load_dotenv, provider_config, public_provider, preflight, percent_prefix_count, run_dataset, export_dataset

DOTENV_PATH = load_dotenv(ROOT)
CONFIG = provider_config(ROOT)
DATASETS = ["devign", "reveal"]
PERCENT = 100.0
RUN_API = True  # this cell sends requests when executed

print("Bundle:", ROOT)
print(".env file:", DOTENV_PATH if DOTENV_PATH else "not found")
print("Provider/model/base URL:", public_provider(CONFIG))
for dataset in DATASETS:
    info = preflight(ROOT, dataset)
    print(info, "selected_prefix_n=", percent_prefix_count(info["n"], PERCENT))


Bundle: D:\JUNIOR\LabThayTho\SHARE_BASELINE_WHITENING_DEVIGN_REVEAL_TEST_CONTEXT8192_20261024
.env file: D:\JUNIOR\LabThayTho\SHARE_BASELINE_WHITENING_DEVIGN_REVEAL_TEST_CONTEXT8192_20261024\.env
Provider/model/base URL: {'provider': 'OpenAI-compatible API', 'model': 'DeepSeek-V4.1-Flash', 'safe_endpoint': 'https://token-api.fpt.ai/v1/chat/completions', 'project': None, 'location': None}
{'dataset': 'devign', 'split': 'test', 'arm': 'baseline_whitening', 'n': 2732, 'max_prompt_utf8_bytes': 8192, 'source_run_id': '1c092d22f44923fc'} selected_prefix_n= 2732
{'dataset': 'reveal', 'split': 'test', 'arm': 'baseline_whitening', 'n': 2274, 'max_prompt_utf8_bytes': 8192, 'source_run_id': 'ee956918e7b19785'} selected_prefix_n= 2274


## Inference and resumable progress

The next cell sends requests for the first `floor(n × PERCENT / 100)` rows of each frozen cohort. It prints per-request percent progress and saves a checkpoint after every response. Increase `PERCENT` to 100 and rerun the cell to finish; completed requests for the same provider, model, endpoint, and prompt are skipped.

If the package `.env` contains all three non-empty variables `LLM_API`, `LLM_MODEL`, and `LLM_BASE_URL`, it uses that OpenAI-compatible API. If all three are absent/empty, it uses Vertex AI with ADC and the configured DeepSeek V4.1 model. A partial API configuration stops before sending a request.


In [7]:
if RUN_API:
    RUN_RESULTS = []
    for dataset in DATASETS:
        result = run_dataset(ROOT, dataset, percent=PERCENT, allow_api=True)
        RUN_RESULTS.append(result)
        print(result)
        print(export_dataset(ROOT, dataset, percent=PERCENT))
else:
    print("RUN_API=False; no API requests were sent")


{'dataset': 'devign', 'run_id': 'abbf356612344183', 'requested_percent': 100.0, 'selected_n': 2732, 'new_requests': 0, 'terminal_requests_after': 2732, 'output': 'D:\\JUNIOR\\LabThayTho\\SHARE_BASELINE_WHITENING_DEVIGN_REVEAL_TEST_CONTEXT8192_20261024\\outputs\\devign\\abbf356612344183', 'provider': {'provider': 'OpenAI-compatible API', 'model': 'DeepSeek-V4.1-Flash', 'safe_endpoint': 'https://token-api.fpt.ai/v1/chat/completions', 'project': None, 'location': None}}
{'dataset': 'devign', 'output': 'D:\\JUNIOR\\LabThayTho\\SHARE_BASELINE_WHITENING_DEVIGN_REVEAL_TEST_CONTEXT8192_20261024\\outputs\\devign\\abbf356612344183', 'selected_prefix_n': 2732, 'status_counts': {'valid': 2732}}
reveal: 1/1610 new requests (0.1%); reveal:test:-7553540414561818627#664 -> valid
reveal: 2/1610 new requests (0.1%); reveal:test:-7248963932691286231#665 -> valid
reveal: 3/1610 new requests (0.2%); reveal:test:9009378236683657160#666 -> valid
reveal: 4/1610 new requests (0.2%); reveal:test:599153350919266

## Output

Each dataset/run folder under `outputs/` contains `predictions.jsonl`, `raw_responses.jsonl`, per-request checkpoints, `run_manifest.json`, and `export_manifest.json`. These files contain record IDs and model outputs, but no ground-truth labels or evaluation metrics.
